# Microsoft SQL Server Access

Write the Iris dataset to Microsoft SQL Server through FreeTDS and read it back.

In [ ]:
import getpass

import pandas as pd

from sklearn.datasets import load_iris

If you do not know the SQL Server port, open a terminal in RStudio and run `tsql -L -H <server-domain>`.

In [ ]:
host = "<server-domain>"
port = 1433
database = "<database>"
user = "<user>"

In [ ]:
df = load_iris(as_frame=True)
df.frame.head()

## SQLAlchemy 
[SQLAlchemy](https://www.sqlalchemy.org/) is a Python library that provides a database abstraction layer so you can connect to different SQL databases.
It works well with `pandas.to_sql()` and `pandas.read_sql()` and can be used with different backends.  

In [ ]:
from sqlalchemy import URL, create_engine

engine = create_engine(
    URL.create(
        "mssql+pyodbc",
        username=user,
        password=getpass.getpass("Database password: "),
        host=host,
        port=port,
        database=database,
        query={"driver": "FreeTDS", "TDS_Version": "7.4"},
    )
)

df.frame.to_sql("iris_example", engine, if_exists="replace", index=False)
pd.read_sql("SELECT * FROM iris_example", engine).head()

## pyodbc cursor

Direct `pyodbc` cursor calls are more verbose, but they show the underlying DB-API flow: open a connection, create a cursor, execute SQL, commit, fetch rows, and close the connection.

Documentation: [pyodbc](https://github.com/mkleehammer/pyodbc/wiki).

In [ ]:
import pyodbc

conn = pyodbc.connect(
    "DRIVER={FreeTDS};"
    f"SERVER={host};"
    f"PORT={port};"
    f"DATABASE={database};"
    f"UID={user};"
    f"PWD={getpass.getpass('Database password: ')};"
    "TDS_Version=7.4"
)

try:
    cursor = conn.cursor()
    cursor.execute("DROP TABLE IF EXISTS iris_example_cursor")
    cursor.execute("""
        CREATE TABLE iris_example_cursor (
            [sepal length (cm)] FLOAT,
            [sepal width (cm)] FLOAT,
            [petal length (cm)] FLOAT,
            [petal width (cm)] FLOAT,
            target INTEGER
        )
    """)
    cursor.executemany("""
        INSERT INTO iris_example_cursor (
            [sepal length (cm)],
            [sepal width (cm)],
            [petal length (cm)],
            [petal width (cm)],
            target
        )
        VALUES (?, ?, ?, ?, ?)
    """, df.frame.itertuples(index=False, name=None))
    conn.commit()

    cursor.execute("SELECT TOP 5 * FROM iris_example_cursor")
    rows = cursor.fetchall()
    columns = [column[0] for column in cursor.description]
    display(pd.DataFrame.from_records(rows, columns=columns))
finally:
    conn.close()